# <center>R3.08 - Probabilités <br>TP3 - Chaînes de Markov et Algorithme du N-grammes<center>


_Tom Ferragut, Thibault Godin_

_IUT de Vannes, BUT Informatique_

In [1]:
import re
import unicodedata
import urllib.request
import random


import matplotlib.pyplot as plt
%matplotlib inline


import numpy as np

# Partie 1 - Chaines de Markov en python (Exercice 1 TD4)

> __Question 1 :__ Définir la matrice de transition $P$ de l'exercice 1 TD4

In [2]:
matriceTransition = np.array([[0,1,0], [0, 0.5, 0.5], [0, 1, 0]])

print(matriceTransition)

[[0.  1.  0. ]
 [0.  0.5 0.5]
 [0.  1.  0. ]]


> __Question 2 :__ Calculer numériquement $P^5$ et en déduire $\mathbb{P}(X_5=3\vert X_0=1)$.

In [3]:
matriceFinale = np.linalg.matrix_power(matriceTransition, 5)[0,2]

print(matriceFinale)

0.3125


> __Question 3 :__ Écrire une fonction `Xn`, prenant en entrée un paramètre `n` et calculant la loi de $X_n$ si $X_0$ suit une loi uniforme.

In [6]:
v0  = 1/3 * np.ones((1,3))

def Xn(n):
    for i in n:
        vn1 = v0@matriceTransition
        return(vn1)



> __Question 4 :__ Que constatez-vous sur le tracé de l'évolution des probabilités (cellule suivante) de $X_n$?

In [8]:
iters = 100
x = np.array(range(iters))
Xnv = np.zeros((iters,3))
print(Xnv[0])
Xnv[0]=v0.copy()
for i in range(1,iters):
    Xnv[i]=Xnv[i-1]@matriceTransition
print(Xnv[1])


Traj=Xnv.transpose()

N = 3;vn1
colors = plt.cm.rainbow(np.linspace(0, 1, N))
for i in range(N):
    plt.plot(x,Traj[i], lw=2, color=colors[i])
plt.legend(loc  ='upper right', ncol=N/2)
plt.show()


[0. 0. 0.]
[0.         0.83333333 0.16666667]


NameError: name 'vn1' is not defined

> __Question 5 :__ Déterminer par le calcul la loi à l'équilibre de la chaîne de Markov, c'est-à-dire la loi $v_{eq}$ telle que $v_{eq} = v_{ep}.P$
>
> Que constatez-vous ?

> __Question 6 :__  Refaire l'exercice avec une autre loi initiale. Que constatez-vous ?

> __Question Bonus :__ À quelle vitesse à lieu la convergence (on utilisera la norme de la différence vectorielle) ?
> Que se passe-t-il si on prend la chaîne de l'exercice 2 ? Et avec une chaîne de période 3 ? Tester avec plusieurs lois initiales.


# Partie 2 - N-grammes
## 2.0 - Introduction


Dans ce TP, nous allons tenter de modéliser une chaîne de Markov permettant de générer un texte automatiquement. Bien que ce procédé soit beaucoup moins puissant que les intelligences artificielles génératives disponibles aujourd'hui, il offre une approche intéressante pour comprendre le fonctionnement des modèles de langage.

Cette méthode s'appelle **N-grammes**, et elle se base sur un texte (ou un corpus de textes) déjà existant. 

---

### Principe des N-grammes

Le principe de fonctionnement est le suivant :

1. **Analyse de texte** : On commence par analyser un texte pour déterminer les mots présents après chaque suite de N mots. Cette étape est essentielle pour établir les relations entre les mots et leurs contextes.

2. **Fréquence d'apparition** : Ensuite, on ordonne chaque mot présent à la suite de N mots par fréquence d'apparition. Cela nous permet de comprendre quelles combinaisons de mots sont les plus courantes et, par conséquent, les plus susceptibles d'apparaître ensemble.

3. **Génération de texte** : Lors de la génération de texte, nous proposons comme mot suivant le mot le plus fréquent après une suite donnée de N mots, ou bien un mot aléatoire pondéré selon sa fréquence d'apparition. Cette méthode nous permet de créer des phrases qui semblent naturelles tout en étant générées de manière algorithmique.

---

### Crédits

- Nous tirons le format .txt du premier tome des *Misérables* de Victor Hugo grâce au projet Gutenberg.
- Ce TP est entièrement constitué du document disponible sur : [Ngram Tutorial GitHub Repository](https://github.com/Elucidation/Ngram-Tutorial/blob/master/NgramTutorial.ipynb).


## 2.1 - Analyse de texte

Dans cette partie sur l'analyse du texte, nous vous fournissons un code permettant de générer une liste contenant chaque mot présente dans le livre _'Les misérables Tome I: Fantine by Victor Hugo'_.

In [9]:
# Find the number links by looking on Project Gutenberg in the address bar for a book.
books = {'Les misérables Tome I: Fantine by Victor Hugo': '17489'}

book = books['Les misérables Tome I: Fantine by Victor Hugo']

url_template = 'https://www.gutenberg.org/cache/epub/%s/pg%s.txt'
url = url_template % (book, book)

# Open the URL and read the content
f = urllib.request.urlopen(url)
txt = f.read().decode('utf-8')  # Decode bytes to string
f.close()

# Extract text between START and END markers
start_marker = "START OF THE PROJECT GUTENBERG"
end_marker = "END OF THE PROJECT GUTENBERG"

# Find the start and end indices for the desired text
start_index = txt.find(start_marker)
end_index = txt.find(end_marker)

if start_index != -1 and end_index != -1:
    # Slice the text to get the desired portion
    # Adding the length of the start_marker to skip to the content after it
    content = txt[start_index + len(start_marker):end_index].strip()
    content = unicodedata.normalize('NFC', content)
    print(f"Extracted content length: {len(content)} characters")
    print(content[:200], '...')  # Print the first 200 characters of the extracted content
else:
    print("Markers not found in the text.")


Extracted content length: 672387 characters
EBOOK LES MISÉRABLES TOME I: FANTINE ***




Produced by www.ebooksgratuits.com and Chuck Greif




Victor Hugo

LES MISÉRABLES

Tome I--FANTINE

(1862)


TABLE DES MATIÈRES

Livr ...


In [107]:
# Diviser le texte en mots, en ignorant les caractères non alphabétiques et en mettant tout en minuscule
words = re.findall(r'\b\w+\b', content)

# Filtrer les chaînes vides résultant de la division
words_upper = list(filter(None, words))  # Convertir l'itérable en liste

    
 # Convertir tous les mots en minuscules
words = [word.lower() for word in words_upper]

# Imprimer la longueur de la liste
print("Dans cet ouvrage, il y a",len(words),"mots.")


Dans cet ouvrage, il y a 119672 mots.


---
Nous définissons maintenant la liste des mots différents, appelés les __2-grammes__.

In [108]:
# Create set of all unique words, this throws away any information about frequency however
gram1 = set(words)

print ("Dans cet ouvrage, il y a",len(gram1),"mots différents, aussi appelés 1-grammes.")

# Instead of printing all the elements in the set, create an iterator and print 20 elements only
gram1_iter = iter(gram1)
print ('Voici 20 exemples de mots présent dans cet ouvrage :',[next(gram1_iter) for i in range(20)])

Dans cet ouvrage, il y a 12232 mots différents, aussi appelés 1-grammes.
Voici 20 exemples de mots présent dans cet ouvrage : ['épître', 'venue', 'lettres', 'trentaine', 'sassenaye', 'moutou', 'brassée', 'vite', 'ajoutaient', 'attribua', 'essayèrent', 'retient', 'seize', 'hauteur', 'pincée', 'marchaient', 'tressaillir', 'battue', 'montreuil', 'foudroyante']


---
Nous allons maintenant lister les ensemble de deux mots successif, aussi appelés les __2-grammes__ (ou bigrammes).

In [109]:
# See the last 10 pairs
for i in range(len(words)-10, len(words)-1):
    print (words[i], words[i+1])

à la
la fosse
fosse publique
publique sa
sa tombe
tombe ressembla
ressembla à
à son
son lit


In [120]:
# Créer des paires de mots consécutifs (bigrammes)
word_pairs = [(words[i], words[i+1]) for i in range(len(words)-1)]
print(len(word_pairs))

# Créer un ensemble des bigrammes uniques
gram2 = set(word_pairs)
print(len(gram2))

# Imprimer 20 éléments de gram2
gram2_iter = iter(gram2)
print([next(gram2_iter) for i in range(20)])


119671
62750
[('comme', 'mes'), ('de', 'mordre'), ('toute', 'bouleversée'), ('la', 'souris'), ('titre', 'de'), ('cela', 'concédé'), ('ne', 'pût'), ('âge', 'son'), ('chasseriez', 'aussi'), ('la', 'bataille'), ('pourtant', 'nous'), ('plus', 'moi'), ('qui', 'reviendra'), ('courant', 'au'), ('semblait', 'voir'), ('plus', 'impénétrable'), ('jour', 'une'), ('sans', 'bagage'), ('notaire', 'sont'), ('lettres', 'elle')]


## 2.2 - Fréquence d'apparition 

Maintenant que nous avons généré la liste des 1-grammes et des bigrammes, nous pouvons mettre en place des compteurs pour mémoriser le nombre d'occurrences de chaque mot.

> __Question 1 :__ À partir de la liste des mots différents `gram1` et la liste de tout les mots `words` générées dans la partie précédente, créer un dictionnaire contenant chaque mot et son nombre d'occurences.
>
>Puis, afficher les 20 mots les plus fréquents dans l'ouvrage de Victor Hugo.

In [121]:
gram1 = dict()

# Remplir gram1 des mots (keys) associés à leur nombre d'occurences

for i in words:
  gram1[i] = gram1.get(i, 0) + 1

# Define a separate function to use for sorting by count
def sort_by_count(item):
    word, count = item
    return -count

# Turn into a list of (word, count) sorted by count from most to least
gram1 = sorted(gram1.items(), key=sort_by_count)

# Print top 20 most frequent words
print(gram1[:20])


[('de', 4470), ('il', 3186), ('la', 3031), ('et', 2945), ('le', 2539), ('l', 2387), ('à', 2321), ('un', 1806), ('les', 1521), ('que', 1349), ('une', 1313), ('d', 1292), ('qui', 1277), ('qu', 1210), ('en', 1190), ('était', 1175), ('dans', 1132), ('est', 1127), ('ce', 1061), ('des', 948)]


> __Question 2 :__ Même question pour les bigrammes, à partir de la liste des couples de mots successifs `gram2` et la liste de tout les mots `words` générées dans la partie précédente, créer un dictionnaire contenant chaque bigrammes et son nombre d'occurences.
>
>Puis, afficher les 20 bigrammes les plus fréquents dans l'ouvrage de Victor Hugo.

In [164]:
bigram_dict = dict()

for i in range(len(words) - 1):
    bigram = (words[i], words[i+1])
    bigram_dict[bigram] = bigram_dict.get(bigram, 0) + 1

def sort_by_count(item):
    bigram, count = item
    return -count

sorted_bigrams = sorted(bigram_dict.items(), key=sort_by_count)

print(sorted_bigrams[:20])

[(('de', 'la'), 583), (('qu', 'il'), 528), (('c', 'est'), 415), (('de', 'l'), 353), (('il', 'y'), 304), (('à', 'la'), 278), (('l', 'évêque'), 236), (('dans', 'la'), 218), (('à', 'l'), 216), (('qu', 'on'), 210), (('d', 'un'), 208), (('il', 'n'), 205), (('c', 'était'), 198), (('il', 'avait'), 194), (('jean', 'valjean'), 193), (('y', 'a'), 193), (('d', 'une'), 190), (('il', 'se'), 181), (('il', 'était'), 180), (('et', 'de'), 169)]


## 2.3 - Génération de texte
### A - Bigrammes (2-grammes)

> __Question 3 :__ Grâce à la bibliothèque `random`, choisir un mot aléatoire dans la liste `words`. Comment calculer la probabilité de tirage de chaque mot ? (Nous ne demandons pas de faire la calcul).

In [113]:
# Choisir un mot au hasard dans la liste words
def randomWords():
    numero = random.randint(1, len(words))
    
    return(words[numero])

randomWords()

'sa'

---
Voici une fonction générant une phrase à partir d'un premier mot. À chaque étape de génération, le mot suivant est celui apparaissant le plus souvent (dans notre corpus) après le dernier mot actuel de notre phrase.

In [148]:
# Générer le dictionnaire des bigrammes où chaque premier mot est associé à une liste de ses successeurs
bigram_dict = dict()

for i in range(len(words) - 1):
    first_word, second_word = words[i], words[i+1]
    if first_word not in bigram_dict:
        bigram_dict[first_word] = []
    bigram_dict[first_word].append(second_word)

def get2GramSentence(start_word, n=20):
    """
    Génère une phrase basée sur un modèle de 2-grammes (bigrammes).
    Le processus commence avec un mot initial et cherche le mot suivant 
    dans les bigrammes déjà générés. Si aucun mot suivant n'est trouvé, 
    la génération s'arrête.

    Paramètres:
    - start_word : Le mot de départ pour la génération de la phrase.
    - n : Le nombre maximum de mots dans la phrase générée.

    Retourne:
    - Une phrase générée basée sur les bigrammes.
    """
    sentence = [start_word]  # Initialiser la phrase avec le mot de départ

    for i in range(n - 1):  # Générer jusqu'à n mots
        current_word = sentence[-1]
        
        # Trouver le mot suivant dans le dictionnaire des bigrammes
        if current_word in bigram_dict:
            next_word = bigram_dict[current_word][0]  # Choisir le premier mot de la liste
        else:
            break  # Si aucun mot suivant n'est trouvé, arrêter la génération
        
        sentence.append(next_word)  # Ajouter le mot suivant à la phrase
    
    return " ".join(sentence)  # Retourner la phrase complète sous forme de chaîne de caractères


>__Question 4 :__ Tester cette fonction sur différents exemples, qu'observez vous ?

In [149]:
print(get2GramSentence("la", 10))

la soeur chapitre i fantine produced by www ebooksgratuits com


---
Pour éviter les problèmes observé dans la méthode précédente, nous allons ajouter un aléatoire dans le choix du mot suivant.

Nous vous fournissons la méthode `weighted_choice`, permettant, comme indiqué, de sélectionner un élément parmi une liste de tuples (élément, poids) en fonction d'une probabilité pondérée.

> __Question 5 :__ En utilisant `weighted_choice`, la liste des bigrammes `gram2` ainsi que la structure de `get2GramSentence`, écrire une méthode `get2GramSentenceRandom`. Cette fonction dois générer une phrase à partir d'un mot de départ, en choisissant chaque mot suivant avec une probabilité dépendant de sa fréquence d'apparition après le bigramme actuel.

In [178]:
def weighted_choice(choices):
    """
    Sélectionne un élément parmi une liste de tuples (élément, poids) en fonction d'une probabilité pondérée.
    
    Paramètres:
    - choices : Une liste de tuples, où chaque tuple est de la forme (élément, poids).
    
    Retourne:
    - Un élément de la liste, choisi en fonction des poids fournis.
    """
    total = sum(w for c, w in choices)  # Calculer la somme des poids
    r = random.uniform(0, total)  # Générer un nombre aléatoire entre 0 et la somme totale des poids
    upto = 0  # Initialiser la somme cumulative
    
    for c, w in choices:
        if upto + w > r:  # Si la somme cumulative dépasse le nombre aléatoire, retourner l'élément
            return c
        upto += w  # Ajouter le poids courant à la somme cumulative
    
    # En cas d'erreur (ce qui ne devrait pas arriver avec les données valides)
    raise ValueError("La fonction weighted_choice n'a pas pu trouver d'élément.")

def get2GramSentenceRandom(start_word, n=50):
    """
    Génère une phrase basée sur un modèle de 2-grammes (bigrammes) avec un choix pondéré.
    Le processus commence avec un mot initial et choisit les mots suivants
    avec une probabilité proportionnelle à leur fréquence d'apparition.
    
    Paramètres:
    - start_word : Le mot de départ pour la génération de la phrase.
    - n : Le nombre maximum de mots dans la phrase générée.
    
    Retourne:
    - Une phrase générée basée sur les bigrammes.
    """
    sentence = [start_word]  # Initialiser la phrase avec le mot de départ
    
    for i in range(n - 1):  # Générer jusqu'à n mots
        
        # Trouver tous les bigrammes possibles commençant par le mot actuel
        
        for i in bigram_dict: # Ne marche pas
            if(bigram_dict[i] == start_word):
                print(bigram_dict[i + 1, 1])

        # Si aucun bigramme n'est trouvé, arrêter la génération
            
        ### TODO ###
        
        # Choisir un bigramme en fonction de la fréquence (poids)
        
        ### TODO ###
    
    return " ".join(sentence)  # Retourner la phrase complète sous forme de chaîne de caractères

Testons votre fonction sur une liste d'exemples :

In [179]:
for word in ['et', 'il', 'elle', 'quand', 'celui', 'jamais', 'je', 'comment']:
    print("Mot de départ :", word)
    # Générer et afficher une phrase aléatoire basée sur les bigrammes avec pondération
    sentence = get2GramSentenceRandom(word, 20)
    print("Phrase en 2-grammes : ", sentence  ,"\n")

Mot de départ : et
Phrase en 2-grammes :  et 

Mot de départ : il
Phrase en 2-grammes :  il 

Mot de départ : elle
Phrase en 2-grammes :  elle 

Mot de départ : quand
Phrase en 2-grammes :  quand 

Mot de départ : celui
Phrase en 2-grammes :  celui 

Mot de départ : jamais
Phrase en 2-grammes :  jamais 

Mot de départ : je
Phrase en 2-grammes :  je 

Mot de départ : comment
Phrase en 2-grammes :  comment 



Voici un test avec une phrase plus longue.

In [15]:
word = 'cela'
print("Mot de départ :", word)
# Générer et afficher une phrase aléatoire basée sur les bigrammes avec pondération
sentence = get2GramSentenceRandom(word, 100)
print("Phrase en 2-grammes : ", sentence  ,"\n")

Mot de départ : cela


NameError: name 'get2GramSentenceRandom' is not defined

> __Question 6 :__ Où sont les marches aléatoires dans tout ça ? Une fois cette marche aléatoire trouver, essayer de définir son ensemble d'états et determiner en quoi consisterait sa matrice de transition.

### B - Trigrammes et plus

Pour aller plus loin nous pouvons lister les __n-grammes__ et les utiliser pour générer notre texte.

In [3]:
def generateNgram(n=1):
    """
    Génère un dictionnaire de N-grammes à partir d'une liste de mots.
    
    Paramètres:
    - n : Le nombre de mots dans chaque N-gramme (doit être supérieur à 0).
    
    Retourne:
    - Une liste de tuples (N-gramme, fréquence) triée par fréquence, de la plus élevée à la plus basse.
    """
    gram = dict()  # Dictionnaire pour stocker les N-grammes et leurs fréquences
    
    # Remplir le dictionnaire N-gram
    for i in range(len(words) - (n - 1)):
        key = tuple(words[i:i + n])  # Créer un N-gramme sous forme de tuple
        if key in gram:  # Vérifier si le N-gramme existe déjà dans le dictionnaire
            gram[key] += 1  # Incrémenter la fréquence
        else:
            gram[key] = 1  # Initialiser la fréquence à 1
    
    # Convertir le dictionnaire en une liste de tuples (N-gramme, fréquence)
    gram = sorted(gram.items(), key=lambda item: -item[1])  # Trier par fréquence de manière décroissante
    return gram

# Générer des trigrammes (3-grammes)
trigram = generateNgram(3)

# Afficher les 20 N-grammes les plus fréquents
print(trigram[:20])


NameError: name 'words' is not defined

> __Question 7 :__ En utilisant `weighted_choice`, la liste des N-grammes  ainsi que la structure de `get2GramSentenceRandom`, écrire une méthode `getNGramSentenceRandom`. Cette fonction dois générer une phrase à partir d'un mot de départ, en choisissant chaque mot suivant avec une probabilité dépendant de sa fréquence d'apparition après le N-gramme.

In [16]:
def getNGramSentenceRandom(gram, word, n=50):
    """
    Génère une phrase aléatoire basée sur un modèle de N-grammes avec un choix pondéré.
    
    Paramètres:
    - gram : La liste des N-grammes et de leurs fréquences.
    - word : Le mot de départ pour la génération de la phrase.
    - n : Le nombre maximum de mots dans la phrase générée.
    """
    
    ### TODO ###

    return #TODO    
        
        
# Générer des phrases avec différents N-grammes
for n in range(2, 10):
    # Générer la liste de N-grammes
    print(f"\nGenerating {n}-gram list...", end=' ')
    ngram = generateNgram(n)
    print("Done")

    # Essayer de générer des phrases avec plusieurs mots de départ
    for word in ['et', 'il', 'elle', 'quand', 'celui', 'jamais', 'je', 'comment']:
        print(f" {word} {n}-gram : \"", end=' ')
        getNGramSentenceRandom(ngram, word, 15)  # Générer une phrase avec le mot de départ
        print("\"")  # Terminer l'affichage de la phrase



Generating 2-gram list... 

NameError: name 'generateNgram' is not defined

In [2]:
# Generate 10gram list
print ("Generating %d-gram list..." % n)
gram10 = generateNgram(10)
print ("Done")

NameError: name 'n' is not defined

In [1]:
# Try out a bunch of sentences
for word in ['et', 'il', 'elle', 'quand', 'celui', 'jamais', 'je', 'comment']:
    print ("  %d-gram: \"" % n)
    getNGramSentenceRandom(ngram, word, 100)
    print ("\"")

NameError: name 'n' is not defined

> __Question 8 :__ Comment pourriez-vous améliorer cette génération ?
> Essayer de les implémenter.
>